# De onde são os autores dos livros do ICMC?

O objetivo é acrescentar um país a cada livro de `dados/livros_icmc.csv`.
Vamos seguir este caminho:

1. Usar o ISBN do catálogo; se faltar, procurar por título e autor.
2. Consultar o livro e pegar seu primeiro autor.
3. Identificar esse autor e procurar cidadania ou nacionalidade declarada.
4. Se esse dado não existir, usar o país de nascimento.

As consultas usam **Open Library, BrasilAPI, Google Books, Wikidata e DBpedia**.
O resultado será uma tabela no próprio notebook e **um único arquivo**:
`dados/livros_autores.csv`. Não há cache, arquivos auxiliares nem mapa.

Um ISBN encontrado por título pode ser de outra edição. Um país obtido pelo
nascimento não é necessariamente uma cidadania. As colunas `origem_isbn` e
`criterio_pais` deixam essas diferenças explícitas. Casos não resolvidos ficam vazios.

## 1. Ler os dados e preparar as consultas

Instale `analise_nacionalidades/requirements.txt` e selecione esse ambiente como kernel.
Execute as células em ordem. Comece com `LIMITE = 5`; use `None` para todos.
Processamos até `TRABALHADORES = 4` livros em paralelo com threads: enquanto uma
consulta espera a rede, outros livros podem avançar. Use `1` para execução sequencial.
Cada livro mantém a mesma ordem de fontes e de fallback.

A pausa é compartilhada por API, em vez de aplicada antes de toda consulta:
Open Library recebe no máximo cerca de uma consulta por segundo; as demais,
até duas. Um erro 429 faz todos os trabalhadores aguardarem o `Retry-After`
daquela API antes de novas consultas, com uma nova tentativa por requisição.
Sem esse cabeçalho, a espera é de 60 segundos.
O ganho depende da latência e dos limites dos serviços; não aumenta a quota da chave.
Não criamos cache nem arquivos extras.

Open Library e BrasilAPI não precisam de chave. Google Books é opcional: para
usá-lo, configure `GOOGLE_BOOKS_API_KEY` no ambiente. Sem chave, essa fonte é pulada.

In [1]:
import os
from getpass import getpass

os.environ["GOOGLE_BOOKS_API_KEY"] = getpass("Cole sua chave do Google Books: ")

In [2]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from email.utils import parsedate_to_datetime
from threading import Lock, local
from urllib.parse import quote, unquote, urlparse
import os
import re
import time
import unicodedata

import pandas as pd
import pycountry
import requests

LIMITE = None  # Troque por 5 para testar somente os primeiros livros.
TRABALHADORES = 6  # Livros processados ao mesmo tempo; use 1 para comparar.
CHAVE_GOOGLE = os.getenv('GOOGLE_BOOKS_API_KEY', '')

RAIZ = Path.cwd()
if not (RAIZ / 'dados/livros_icmc.csv').exists():
    RAIZ = RAIZ.parent  # Permite executar de dentro de analise_nacionalidades/.
livros = pd.read_csv(RAIZ / 'dados/livros_icmc.csv', dtype=str, keep_default_na=False)
livros = livros.head(LIMITE) if LIMITE is not None else livros

# A pausa vale por API, compartilhada entre todos os trabalhadores.
# Open Library: pouco menos de 1 consulta/s; outras fontes: até 2/s.
INTERVALOS = {'openlibrary.org': 1.1}
proxima_consulta = {}
controle_consultas = Lock()
por_thread = local()

def aguardar_api(dominio):
    """Libera uma consulta sem bloquear o uso das outras APIs."""
    while True:
        with controle_consultas:
            agora = time.monotonic()
            espera = proxima_consulta.get(dominio, 0) - agora
            if espera <= 0:
                proxima_consulta[dominio] = agora + INTERVALOS.get(dominio, 0.5)
                return
        time.sleep(espera)  # Fora do lock: outras APIs podem continuar.

def consultar(url, parametros=None, cabecalhos=None):
    """Consulta JSON; se a API limitar o acesso (429), espera e tenta mais uma vez."""
    # Cada trabalhador reutiliza sua conexão, sem compartilhar uma Session mutável.
    if not hasattr(por_thread, 'sessao'):
        por_thread.sessao = requests.Session()
        por_thread.sessao.headers['User-Agent'] = 'LivrosICMC/1.0 (https://github.com/LVinaud/LivrosICMC)'
    dominio = urlparse(url).hostname
    for tentativa in range(2):
        aguardar_api(dominio)
        try:
            resposta = por_thread.sessao.get(url, params=parametros, headers=cabecalhos, timeout=20)
            if resposta.status_code == 429:
                # Retry-After pode informar segundos ou uma data HTTP.
                aviso = resposta.headers.get('Retry-After', '')
                try:
                    espera = float(aviso)
                except ValueError:
                    try:
                        espera = parsedate_to_datetime(aviso).timestamp() - time.time()
                    except (ValueError, TypeError, OverflowError):
                        espera = 60
                with controle_consultas:
                    proxima_consulta[dominio] = max(
                        proxima_consulta.get(dominio, 0), time.monotonic() + max(1, espera))
                if tentativa == 0:
                    print(f'{dominio}: limite de consultas; aguardando para tentar novamente.', flush=True)
                    continue
            resposta.raise_for_status()
            return resposta.json()
        except (requests.RequestException, ValueError) as erro:
            print(f'Consulta indisponível: {url.split("?")[0]} ({type(erro).__name__})', flush=True)
            return {}
    return {}

print(f'{len(livros)} registros para analisar.')
livros[['titulo', 'autores', 'isbn']].head()

84 registros para analisar.


,titulo,autores,isbn
0,Eu sou Malala a história da garota que defende...,Malala Yousafzai 1997- | Caroline Chang; Denis...,9788535923438
1,A cooperação universidade-escola para a formaç...,Emerson de Pietri | Vinício de Macedo Santos; ...,9788578613440
2,Conversa com professores do fundamental à pós-...,Newton Cesar Balzan,9788524923500
3,Iniciação a lógica matemática,Edgard de Alencar Filho 1913-,9788521304036
4,Negócios digitais aprenda a usar o real poder ...,Alan Pakes,9788545200598


## 2. Pequenas funções para comparar os dados

O CSV pode ter vários ISBNs separados por `|`. Validamos os dígitos e convertemos
ISBN-10 para ISBN-13. Para comparar nomes, ignoramos acentos, datas e pontuação,
mas não aceitamos uma simples semelhança de sobrenome.

In [3]:
def limpar_nome(nome):
    nome = re.sub(r'\b\d{4}\s*[-–]\s*(?:\d{4})?|\bauthor\b|\bet\s+al\.?', '', nome, flags=re.I)
    return re.sub(r'\([^)]*\)', '', nome).strip(' .,')

def palavras(texto):
    texto = unicodedata.normalize('NFKD', texto.casefold())
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return re.findall(r'[a-z0-9]+', texto)

def nomes_iguais(a, b):
    x, y = palavras(limpar_nome(a)), palavras(limpar_nome(b))
    if not x or not y:
        return False
    if sorted(x) == sorted(y):
        return True
    # Exemplo aceito: Yuval N. Harari e Yuval Noah Harari.
    return (len(x) == len(y) and sum(t in y and len(t) > 1 for t in x) >= 2
            and all(u == v or (len(u) == 1 and v.startswith(u))
                    or (len(v) == 1 and u.startswith(v)) for u, v in zip(x, y)))

def isbn13(base):
    soma = sum(int(c) * (1 if i % 2 == 0 else 3) for i, c in enumerate(base))
    return base + str(-soma % 10)

def validar_isbn(texto):
    numero = re.sub(r'[\s\-‐‑–—]', '', texto.upper())
    if re.fullmatch(r'[0-9]{9}[0-9X]', numero):
        soma = sum((10-i) * (10 if c == 'X' else int(c)) for i, c in enumerate(numero))
        return isbn13('978' + numero[:9]) if soma % 11 == 0 else ''
    if re.fullmatch(r'(978|979)[0-9]{10}', numero) and not numero.startswith('9790'):
        return numero if isbn13(numero[:12]) == numero else ''
    return ''

def titulos_iguais(a, b):
    ignorar = {'a', 'o', 'de', 'da', 'do', 'e', 'the', 'of', 'and'}
    x, y = [t for t in palavras(a) if t not in ignorar], [t for t in palavras(b) if t not in ignorar]
    if not x or not y:
        return False
    if x == y:
        return True
    # Alguns catálogos juntam título e subtítulo sem pontuação.
    return min(len(x), len(y)) >= 2 and (x[:len(y)] == y or y[:len(x)] == x
            or 2 * len(set(x) & set(y)) / (len(set(x)) + len(set(y))) >= 0.85)

## 3. Encontrar o livro e seu ISBN

Com ISBN, tentamos **Open Library → BrasilAPI → Google Books**. A BrasilAPI
agrega provedores como CBL e Mercado Editorial e ajuda com livros nacionais.

Sem ISBN, buscamos por título + autor em Open Library e Google Books. Exigimos
que ambos sejam compatíveis antes de usar um ISBN retornado. Se houver vários,
usamos um deles para a pesquisa do autor, sem afirmar qual é a edição do catálogo.

In [4]:
def buscar_openlibrary(isbn='', titulo='', autor=''):
    parametros = {'fields': 'key,title,author_name,author_key,isbn', 'limit': 20}
    if isbn:
        parametros['q'] = 'isbn:' + isbn
    else:
        termos = [p for p in palavras(titulo) if p not in {'a', 'o', 'de', 'da', 'do', 'e', 'para', 'the', 'of'}]
        nomes = palavras(limpar_nome(autor))
        if not termos or not nomes:
            return {}
        autor_busca = nomes[0] + ' ' + nomes[-1] if len(nomes) > 1 else nomes[0]
        parametros.update(title=' '.join(termos[:5]), author=autor_busca)
    dados = consultar('https://openlibrary.org/search.json', parametros)
    encontrados = []
    for obra in dados.get('docs', []):
        nomes = obra.get('author_name', [])
        codigos = [validar_isbn(str(i)) for i in obra.get('isbn', [])]
        codigos = [i for i in codigos if i]
        if not nomes or not codigos or (isbn and isbn not in codigos):
            continue
        if not isbn and not (titulos_iguais(titulo, obra['title']) and nomes_iguais(autor, nomes[0])):
            continue
        # Uma pessoa pode ter registros duplicados na Open Library.
        ids = [id for id, nome in zip(obra.get('author_key', []), nomes) if nomes_iguais(nome, nomes[0])]
        encontrados.append(dict(isbn=isbn or codigos[0], titulo=obra['title'], autor=limpar_nome(nomes[0]),
                                ids_openlibrary=ids, fonte_livro='Open Library'))
    if not encontrados:
        return {}
    primeiro = encontrados[0]
    # Une registros duplicados do primeiro autor, sem incluir os coautores.
    for outro in encontrados[1:]:
        if nomes_iguais(primeiro['autor'], outro['autor']):
            primeiro['ids_openlibrary'].extend(outro['ids_openlibrary'])
    return primeiro

def buscar_google(isbn='', titulo='', autor=''):
    if not CHAVE_GOOGLE:
        return {}  # Fonte opcional: evita repetir consultas sem quota/chave.
    busca = 'isbn:' + isbn if isbn else titulo + ' ' + autor
    dados = consultar('https://www.googleapis.com/books/v1/volumes', {'q': busca, 'maxResults': 10},
                      {'X-Goog-Api-Key': CHAVE_GOOGLE})
    for item in dados.get('items', []):
        livro = item.get('volumeInfo', {})
        nomes = livro.get('authors', [])
        codigos = [validar_isbn(i['identifier']) for i in livro.get('industryIdentifiers', [])]
        codigos = [i for i in codigos if i]
        if not nomes or not codigos or (isbn and isbn not in codigos):
            continue
        if not isbn and not (titulos_iguais(titulo, livro['title']) and nomes_iguais(autor, nomes[0])):
            continue
        return dict(isbn=isbn or codigos[0], titulo=livro['title'], autor=limpar_nome(nomes[0]),
                    ids_openlibrary=[], fonte_livro='Google Books')
    return {}

def buscar_brasilapi(isbn):
    if not isbn.startswith(('97885', '97865')):
        return {}  # Provedores nacionais: grupos brasileiros 85 e 65.
    dados = consultar(f'https://brasilapi.com.br/api/isbn/v1/{isbn}')
    if validar_isbn(str(dados.get('isbn', ''))) == isbn and dados.get('authors'):
        return dict(isbn=isbn, titulo=dados.get('title', ''), autor=limpar_nome(dados['authors'][0]),
                    ids_openlibrary=[], fonte_livro='BrasilAPI/' + dados.get('provider', ''))
    return {}

def encontrar_livro(linha):
    codigos = list(dict.fromkeys(validar_isbn(i) for i in linha['isbn'].split('|')))
    for isbn in filter(None, codigos):
        livro = buscar_openlibrary(isbn) or buscar_brasilapi(isbn) or buscar_google(isbn)
        if livro:
            return livro, 'catalogo'
    # Também ajuda quando o catálogo tem ISBN, mas nenhuma API o encontrou.
    autor = linha['autores'].split('|')[0].split(';')[0].strip()
    livro = buscar_openlibrary(titulo=linha['titulo'], autor=autor)
    livro = livro or buscar_google(titulo=linha['titulo'], autor=autor)
    return livro, 'busca_titulo_edicao_nao_confirmada' if livro else ''

## 4. Identificar o primeiro autor

Os metadados do livro informam o primeiro autor e, às vezes, seu ID na Open Library.
Consultamos esses registros para obter um ID Wikidata ou um link explícito para
Wikipedia, que permitirá consultar a DBpedia.

Se o registro estiver incompleto, tentamos o mesmo livro por título + autor na
Open Library. Não escolhemos uma pessoa por uma busca solta de nome. Se os IDs
apontarem para pessoas diferentes, marcamos o caso para revisão.

In [5]:
def identificar_autor(livro):
    ids = livro['ids_openlibrary'][:]
    qids, links = set(), set()
    for tentativa in range(2):
        for id in dict.fromkeys(ids):
            dados = consultar(f'https://openlibrary.org/authors/{id}.json')
            if not nomes_iguais(livro['autor'], dados.get('name', '')):
                continue
            qid = (dados.get('remote_ids') or {}).get('wikidata') or []
            qids.update(q for q in ([qid] if isinstance(qid, str) else qid) if re.fullmatch(r'Q[0-9]+', q))
            if dados.get('wikipedia'):
                links.add(dados['wikipedia'])
            links.update(link.get('url', '') for link in (dados.get('links') or [])
                         if urlparse(link.get('url', '')).hostname == 'en.wikipedia.org')
        if qids or links or tentativa == 1:
            break
        outro = buscar_openlibrary(titulo=livro['titulo'], autor=livro['autor'])
        ids = [id for id in outro.get('ids_openlibrary', []) if id not in ids]
    if len(qids) > 1:
        return '', '', 'autor_ambiguo'
    link = next(iter(links)) if len(links) == 1 else ''
    return next(iter(qids), ''), link, ''

## 5. Procurar o país

Usamos **Wikidata → DBpedia**, nesta prioridade:

1. Cidadania (P27) ou nacionalidade declarada na DBpedia.
2. Nascimento diretamente em um país.
3. País associado à cidade/local de nascimento (P17 ou `dbo:country`).

Antes de usar o Wikidata, conferimos nome, tipo “ser humano” e o ano de nascimento
quando ele existir no catálogo. DBpedia é consultada apenas por um link explícito
do autor. Não inventamos um país a partir do nome, idioma ou editora.

Preservamos múltiplas cidadanias e países históricos. Nascimento é uma aproximação,
não uma cidadania. O país associado a um lugar reflete os dados da API, sem reconstruir
fronteiras na data do nascimento. Quando não houver informação, deixamos o país vazio.

In [6]:
def entidade_wikidata(qid):
    dados = consultar(f'https://www.wikidata.org/wiki/Special:EntityData/{qid}.json')
    entidades = list(dados.get('entities', {}).values())
    return entidades[0] if entidades else {}

def valores_wikidata(dados, propriedade):
    return [d['mainsnak']['datavalue']['value'] for d in dados.get('claims', {}).get(propriedade, [])
            if d.get('rank') != 'deprecated' and d.get('mainsnak', {}).get('snaktype') == 'value']

def rotulo(dados):
    for idioma in ['pt', 'en']:
        if idioma in dados.get('labels', {}):
            return dados['labels'][idioma]['value']
    return ''

def pais_wikidata(qid, dados=None):
    dados = entidade_wikidata(qid) if dados is None else dados
    if not rotulo(dados):
        return None
    iso = next(iter(valores_wikidata(dados, 'P298')), '')
    if valores_wikidata(dados, 'P576') or not pycountry.countries.get(alpha_3=iso):
        iso = ''  # Não converter um Estado histórico em um país atual.
    return dict(pais=rotulo(dados), iso3=iso)

DBPEDIA = 'http://dbpedia.org/'
def dados_dbpedia(uri):
    if not uri.startswith(DBPEDIA + 'resource/'):
        return {}
    nome = unquote(uri.split('/resource/')[1])
    return consultar('https://dbpedia.org/data/' + quote(nome, safe='') + '.json').get(uri, {})

def valores_dbpedia(dados, propriedade):
    return [v['value'] for v in dados.get(DBPEDIA + 'ontology/' + propriedade, []) if v.get('type') == 'uri']

def pais_dbpedia(uri, dados=None):
    dados = dados_dbpedia(uri) if dados is None else dados
    tipos = [v['value'] for v in dados.get('http://www.w3.org/1999/02/22-rdf-syntax-ns#type', [])]
    if DBPEDIA + 'ontology/Country' not in tipos:
        return None
    nome = unquote(uri.rsplit('/', 1)[-1]).replace('_', ' ')
    try:
        iso = pycountry.countries.lookup(nome).alpha_3
    except LookupError:
        iso = ''
    if any('Former' in t or 'Historical' in t for t in tipos):
        iso = ''
    return dict(pais=nome, iso3=iso)

def encontrar_pais(autor, qid, wikipedia, autor_catalogo):
    pessoa = entidade_wikidata(qid) if qid else {}
    if pessoa:
        nomes = [v['value'] for v in pessoa.get('labels', {}).values()]
        nomes += [v['value'] for lista in pessoa.get('aliases', {}).values() for v in lista]
        humano = any(v.get('id') == 'Q5' for v in valores_wikidata(pessoa, 'P31'))
        if not humano or not any(nomes_iguais(autor, nome) for nome in nomes):
            return [], 'autor_nao_confirmado', ''
        ano = re.search(r'\b(\d{4})\s*[-–]', autor_catalogo) if nomes_iguais(autor, autor_catalogo) else None
        datas = valores_wikidata(pessoa, 'P569')
        if ano and datas and not any(d['time'].startswith('+' + ano.group(1)) for d in datas):
            return [], 'nascimento_divergente', ''
        cidadanias = valores_wikidata(pessoa, 'P27')
        if cidadanias:
            paises = []
            for cidadania in cidadanias:
                pais = pais_wikidata(cidadania['id'])
                if pais:
                    paises.append(pais)
            return paises, 'cidadania', 'Wikidata'
        titulo_wiki = pessoa.get('sitelinks', {}).get('enwiki', {}).get('title', '')
        if titulo_wiki:
            wikipedia = 'https://en.wikipedia.org/wiki/' + titulo_wiki

    # DBpedia reaproveita o link explícito para a página do autor na Wikipedia.
    uri = ''
    if urlparse(wikipedia).hostname == 'en.wikipedia.org' and '/wiki/' in wikipedia:
        uri = DBPEDIA + 'resource/' + unquote(wikipedia.split('/wiki/')[1]).replace(' ', '_')
    alternativa = dados_dbpedia(uri) if uri else {}
    for propriedade in ['citizenship', 'nationality']:
        paises = []
        for uri_pais in valores_dbpedia(alternativa, propriedade):
            pais = pais_dbpedia(uri_pais)
            if pais:
                paises.append(pais)
        if paises:
            criterio = 'cidadania' if propriedade == 'citizenship' else 'nacionalidade_declarada'
            return paises, criterio, 'DBpedia'

    # Sem cidadania/nacionalidade: o próximo recurso é o nascimento.
    paises = []
    for local in valores_wikidata(pessoa, 'P19'):
        lugar = entidade_wikidata(local['id'])
        if valores_wikidata(lugar, 'P298'):
            paises.append(pais_wikidata(local['id'], lugar))
        else:
            paises += [pais_wikidata(p['id']) for p in valores_wikidata(lugar, 'P17')]
    if any(paises):
        return [p for p in paises if p], 'pais_nascimento', 'Wikidata'
    for local in valores_dbpedia(alternativa, 'birthPlace'):
        lugar = dados_dbpedia(local)
        direto = pais_dbpedia(local, lugar)
        paises += [direto] if direto else [pais_dbpedia(u) for u in valores_dbpedia(lugar, 'country')]
    return [p for p in paises if p], 'pais_nascimento' if any(paises) else '', 'DBpedia' if any(paises) else ''

## 6. Juntar tudo e salvar um único resultado

Uma linha de saída corresponde a uma linha do catálogo. `primeiro_autor` vem
da API do livro; `autor_catalogo` permite compará-lo com o nome original.
Se um autor tiver mais de um país, os valores são separados por `|`.
O progresso mostra os livros na ordem de conclusão, mas a tabela e o CSV preservam
a ordem do catálogo. Ao final, mostramos o tempo total para facilitar a comparação.

Leia `situacao` para saber o que foi resolvido. “Não encontrado” também pode
significar que a base tem pouca cobertura ou que uma requisição falhou.
Não substituímos as lacunas por palpites. Execute apenas esta célula novamente
se quiser repetir o processamento inteiro; cada execução refaz as consultas.

In [7]:
def analisar_livro(linha):
    autor_catalogo = linha['autores'].split('|')[0].split(';')[0].strip()
    resultado = dict(id=linha['id'], titulo=linha['titulo'], isbn='', origem_isbn='',
        autor_catalogo=autor_catalogo, primeiro_autor='', autor_id='', pais='', pais_iso3='',
        criterio_pais='', fonte_pais='', fonte_livro='', situacao='livro_ou_isbn_nao_encontrado')
    livro, origem = encontrar_livro(linha)
    if not livro:
        return resultado
    resultado.update(isbn=livro['isbn'], origem_isbn=origem,
                     primeiro_autor=livro['autor'], fonte_livro=livro['fonte_livro'])
    qid, wikipedia, erro = identificar_autor(livro)
    resultado['autor_id'] = qid or wikipedia
    if erro or not (qid or wikipedia):
        resultado['situacao'] = erro or 'autor_sem_identificador'
        return resultado
    paises, criterio, fonte = encontrar_pais(livro['autor'], qid, wikipedia, autor_catalogo)
    # Remove repetições sem descartar autores com mais de um país.
    paises_unicos = []
    for pais in paises:
        if pais not in paises_unicos:
            paises_unicos.append(pais)
    paises = paises_unicos
    resultado.update(pais=' | '.join(p['pais'] for p in paises),
        pais_iso3=' | '.join(p['iso3'] for p in paises), criterio_pais=criterio, fonte_pais=fonte,
        situacao='encontrado' if paises else (criterio if criterio in {'autor_nao_confirmado', 'nascimento_divergente'} else 'pais_nao_encontrado'))
    return resultado

inicio = time.perf_counter()
linhas = livros.to_dict('records')
resultados = [None] * len(linhas)
with ThreadPoolExecutor(max_workers=TRABALHADORES) as executor:
    tarefas = {executor.submit(analisar_livro, linha): i for i, linha in enumerate(linhas)}
    for concluidos, tarefa in enumerate(as_completed(tarefas), 1):
        i = tarefas[tarefa]
        resultados[i] = tarefa.result()  # Preserva a ordem original do CSV.
        print(f'{concluidos}/{len(linhas)} concluídos: {linhas[i]["titulo"][:65]}', flush=True)

print(f'Tempo total: {time.perf_counter() - inicio:.1f} segundos.')

resultado = pd.DataFrame(resultados)
print('Registros por situação:')
print(resultado['situacao'].value_counts())
display(resultado)

if LIMITE is None:
    resultado.to_csv(RAIZ / 'dados/livros_autores.csv', index=False)
    print('Salvo em dados/livros_autores.csv')
else:
    print('Execução de teste: resultado apenas na tela, sem sobrescrever o CSV completo.')

1/84 concluídos: Aforismos para a sabedoria de vida
Consulta indisponível: https://www.googleapis.com/books/v1/volumes (HTTPError)
Consulta indisponível: https://brasilapi.com.br/api/isbn/v1/9788536418667 (HTTPError)
2/84 concluídos: Conversa com professores do fundamental à pós-graduação
3/84 concluídos: A cooperação universidade-escola para a formação inicial de profe
4/84 concluídos: E se... os nazistas tivessem ganho a segunda guerra mundial? Jesu
5/84 concluídos: 1889 como um imperador cansado, um marechal vaidoso e um professo
6/84 concluídos: Iniciação a lógica matemática
7/84 concluídos: A fúria dos reis
8/84 concluídos: A história da matematização da natureza
Consulta indisponível: https://www.googleapis.com/books/v1/volumes (HTTPError)
9/84 concluídos: Negócios digitais aprenda a usar o real poder da Internet nos seu
Consulta indisponível: https://www.googleapis.com/books/v1/volumes (HTTPError)
10/84 concluídos: Contributions to nonlinear elliptic equations and systems a trib

,id,titulo,isbn,origem_isbn,autor_catalogo,primeiro_autor,autor_id,pais,pais_iso3,criterio_pais,fonte_pais,fonte_livro,situacao
0,dedupmrg280893984,Eu sou Malala a história da garota que defende...,9788535923438,catalogo,Malala Yousafzai 1997-,Malala Yousafzai,Q32732,Paquistão,PAK,cidadania,Wikidata,Open Library,encontrado
1,dedupmrg347336837,A cooperação universidade-escola para a formaç...,,,Emerson de Pietri,,,,,,,,livro_ou_isbn_nao_encontrado
2,dedupmrg281037225,Conversa com professores do fundamental à pós-...,9788524923906,busca_titulo_edicao_nao_confirmada,Newton Cesar Balzan,Newton Cesar Balzan,,,,,,Google Books,autor_sem_identificador
3,dedupmrg280256655,Iniciação a lógica matemática,9788521304036,catalogo,Edgard de Alencar Filho 1913-,"ALENCAR FILHO, EDGARD DE",,,,,,BrasilAPI/cbl,autor_sem_identificador
4,usp_base01002801906,Negócios digitais aprenda a usar o real poder ...,,,Alan Pakes,,,,,,,,livro_ou_isbn_nao_encontrado
...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,dedupmrg155448848,Probabilidade e variáveis aleatórias,9788531409455,catalogo,Marcos Nascimento Magalhães,Marcos Nascimento Magalhães,,,,,,BrasilAPI/cbl,autor_sem_identificador
80,usp_base01002747232,"Robotics Joint Conference on Robotics, LARS 20...",9783662481332,catalogo,"Joint Conference on Robotics (2014 São Carlos,...",Fernando S. Osório,,,,,,Open Library,autor_sem_identificador
81,usp_base04002720259,Álgebra comutativa em quatro movimentos,,,Eduardo Tengan,,,,,,,,livro_ou_isbn_nao_encontrado
82,dedupmrg280879258,"Pesquisa social teoria, método e criatividade",9788532611451,catalogo,Maria Cecília de Souza Minayo,Maria Cecília de Souza Minayo,,,,,,Open Library,autor_sem_identificador


Salvo em dados/livros_autores.csv


## 7. Resumo dos países encontrados

Contamos quantos registros receberam pelo menos um país e quantos ficaram sem país.
A tabela mostra o número de registros associado a cada país, do maior para o menor.

Um registro com vários países conta uma vez em cada um deles. Por isso, a soma da
tabela pode ser maior que o total de registros com país. Países históricos continuam
com seus nomes originais. Este resumo usa o resultado já obtido, sem novas consultas
às APIs e sem gerar outro CSV.


In [ ]:
# Separa os países de registros com múltiplas cidadanias.
paises = resultado['pais'].fillna('').str.split('|', regex=False).explode().str.strip()
paises = paises[paises.ne('')].rename_axis('registro').reset_index(name='pais')
paises = paises.drop_duplicates(['registro', 'pais'])  # Um país conta uma vez por registro.

total = len(resultado)
com_pais = paises['registro'].nunique()
percentual = 100 * com_pais / total if total else 0
print(f'Registros com país: {com_pais} de {total} ({percentual:.1f}%)')
print(f'Registros sem país: {total - com_pais}')

contagem_paises = paises['pais'].value_counts().rename_axis('pais').reset_index(name='registros')
contagem_paises = contagem_paises.sort_values(
    ['registros', 'pais'], ascending=[False, True], ignore_index=True)
display(contagem_paises)


## Referências para continuar o trabalho

- [Open Library: busca de livros](https://openlibrary.org/dev/docs/api/search)
  e [registros de autores](https://openlibrary.org/dev/docs/api/authors).
- [BrasilAPI: consulta de ISBN](https://brasilapi.com.br/docs).
- [Google Books: buscas por ISBN e texto](https://developers.google.com/books/docs/v1/using).
- [Wikidata: acesso aos dados](https://www.mediawiki.org/wiki/Wikibase/EntityData).
- [DBpedia: dados interligados](https://www.dbpedia.org/resources/linked-data/).

Limites: o primeiro autor depende dos metadados da API; ligações entre bases podem
estar incompletas; cidadania e nascimento não são equivalentes. DBpedia e Wikidata
podem compartilhar a Wikipedia como fonte. Um país histórico fica com nome, mas
sem ISO atual. Antes de mapear, decida como contar autores com vários países.